# Práctica 6 - Preprocesamiento de datos para Machine Learning
**Sistemas de Aprendizaje Automático · Sesión 6**   
**Dataset:** `clientes_ecommerce.csv` · **Modalidad:** individual

## Contexto
Trabajas en una empresa de comercio electrónico que quiere predecir si un cliente comprará el próximo mes. El dataset ya se exploró en las sesiones 4 y 5. Ahora debes transformarlo para que esté listo para entrenar un modelo de clasificación binaria.

**No entrenes modelos en esta práctica.** Completa los bloques `TODO` y responde las preguntas en las celdas Markdown de respuesta.

### Objetivos
- Detectar ausentes y anomalías
- Elegir estrategias de imputación
- Codificar variables nominales y ordinales
- Comparar escalados
- Separar entrenamiento/prueba
- Construir un `Pipeline` con `ColumnTransformer`
- Evitar data leakage
- Verificar las matrices resultantes

> Coloca `clientes_ecommerce.csv` en la misma carpeta que este notebook. Usa scikit-learn 1.2 o posterior para `sparse_output=False`.

## Parte 1 - Diagnóstico del dataset
### Ejercicio 1. Carga y estructura
Carga el CSV, muestra 10 filas, dimensiones, tipos de datos, información general y número de registros duplicados.

In [1]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# TODO: cargar clientes_ecommerce.csv en df
df = pd.read_csv("clientes_ecommerce.csv")
# TODO: mostrar 10 primeras filas
df.head(10)
# TODO: consultar shape, dtypes, info y número de duplicados
print(df.shape)
print(df.dtypes)
print(df.info())
print("Filas duplicadas: ", df.duplicated().sum())

(400, 12)
cliente_id              int64
edad                    int64
antiguedad_meses      float64
ingresos_anuales      float64
gasto_mensual         float64
num_compras             int64
devoluciones            int64
dispositivo               str
zona                      str
nivel_fidelidad           str
satisfaccion            int64
compra_proximo_mes        str
dtype: object
<class 'pandas.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   cliente_id          400 non-null    int64  
 1   edad                400 non-null    int64  
 2   antiguedad_meses    395 non-null    float64
 3   ingresos_anuales    396 non-null    float64
 4   gasto_mensual       397 non-null    float64
 5   num_compras         400 non-null    int64  
 6   devoluciones        400 non-null    int64  
 7   dispositivo         400 non-null    str    
 8   zona                400 non

**Responde:** ¿Cuántas muestras y columnas hay? ¿Cuál es el target? ¿Qué columna identifica a cada cliente? ¿Hay filas duplicadas?

**Respuesta:**

- El dataset tiene 400 filas y 12 columnas.
- El target es compra_proximo_mes, con valores No / Sí (clasificación binaria).
- La columna que identifica a cada cliente es cliente_id (no tiene valores repetidos).
- No hay filas duplicadas (df.duplicated().sum() devuelve 0).

### Ejercicio 2. Valores ausentes
Calcula el número de ausentes por columna, el porcentaje y el número de filas con al menos un ausente. Completa la tabla.

In [3]:
# TODO: ausentes por columna
ausentes = df.isna().sum()
# TODO: porcentaje de ausentes por columna
porcentaje = df.isna().mean().mul(100)
# TODO: filas con algún valor ausente
filas_incompletas = df.isna().any(axis=1).sum()

display(pd.DataFrame({"ausentes": ausentes, "porcentaje": porcentaje.round(2)}))
print("Filas con al menos un ausente:", filas_incompletas)

,ausentes,porcentaje
cliente_id,0,0.00
edad,0,0.00
antiguedad_meses,5,1.25
ingresos_anuales,4,1.00
gasto_mensual,3,0.75
num_compras,0,0.00
devoluciones,0,0.00
dispositivo,0,0.00
zona,0,0.00
nivel_fidelidad,0,0.00


Filas con al menos un ausente: 12


| Variable | Nº ausentes | Porcentaje |
|---|---:|---:|
| `antiguedad_meses` | 5 | 1,25% |
| `ingresos_anuales` | 4 | 1,00% |
| `gasto_mensual` | 3 | 0,75% |

**¿Eliminarías todas las filas incompletas? ¿Por qué?**

Respuesta: No. Hay 12 filas con algún ausente (un 3 % del dataset) y en cada una falta solo un valor, el resto de columnas son válidas. Eliminarlas supondría perder información útil sin necesidad, y además podría sesgar la muestra si los ausentes no son aleatorios. Como el porcentaje de ausentes por columna es muy bajo, es mejor imputarlos.

### Ejercicio 3. Edad sospechosa
Localiza edades superiores a 120 años, muestra `cliente_id` y `edad` y consulta mínimo y máximo.

In [4]:
# TODO: mínimo y máximo de edad
print("Edad mínima:", df["edad"].min())
print("Edad máxima:", df["edad"].max())
# TODO: localizar registros con edad > 120
df.loc[df["edad"] > 120, ["cliente_id", "edad"]]

Edad mínima: 18
Edad máxima: 195


,cliente_id,edad
137,20138,195


**Responde:** ¿Qué harías con esa edad? ¿Por qué no deberías cambiar 195 por 95 sin evidencia?

- Una edad de 195 años es imposible, así que la trataría como un error de dato: la marcaría como valor ausente y después la imputaría igual que el resto de ausentes. No eliminaría la fila entera, porque el resto de sus datos sí son válidos.
- No debería cambiarla por 95 porque eso es inventarse el dato: suponer que sobra un 1 es solo una hipótesis. El valor real podría ser 19, 59 o cualquier otro. Sin una fuente que lo confirme, corregirlo a ojo mete información falsa en el dataset como si fuera cierta.

## Parte 2 - Variables y división
### Ejercicio 4. Clasificación de columnas
Clasifica las columnas según su naturaleza.

| Tipo | Columnas |
|---|---|
| Numéricas | `edad`, `antiguedad_meses`, `ingresos_anuales`, `gasto_mensual`, `num_compras`, `devoluciones`, `satisfaccion` |
| Categóricas nominales | dispositivo, zona |
| Categóricas ordinales | nivel_fidelidad |
| Identificador | cliente_id |
| Target | compra_proximo_mes`|

**Responde:** ¿Por qué `zona` es nominal y `nivel_fidelidad` ordinal? ¿Por qué no incluirías `cliente_id` en el modelo?

Respuesta:

* zona (Norte, Sur, Este, Oeste, Centro) es nominal porque sus categorías no tienen ningún orden natural: ninguna zona es "mayor" o "mejor" que otra.
* nivel_fidelidad es ordinal porque sus categorías sí siguen un orden lógico: Bronce < Plata < Oro < Platino.
* cliente_id es solo un identificador único asignado a cada cliente; su valor no tiene relación con el comportamiento de compra. Si se incluyera, el modelo podría aprender patrones falsos (o memorizar clientes concretos) y no generalizaría a clientes nuevos.

*Nota:* satisfaccion (1-5) es conceptualmente ordinal, pero en esta práctica se tratará como numérica.


### Ejercicio 5. Separar X e y
Excluye de X el identificador y el target. Codifica `No` como 0 y `Sí` como 1. Comprueba dimensiones y distribución de y.

In [5]:
# TODO: X = ... (sin cliente_id ni compra_proximo_mes)
X = df.drop(columns=["cliente_id", "compra_proximo_mes"])
# TODO: y = ... (mapear No:0, Sí:1)
y = df["compra_proximo_mes"].map({"No": 0, "Sí": 1})
# TODO: X.shape, y.shape, y.value_counts()
print("X:", X.shape)
print("y:", y.shape)
print(y.value_counts())

X: (400, 10)
y: (400,)
compra_proximo_mes
0    246
1    154
Name: count, dtype: int64


**¿Cuántas características predictoras iniciales hay?**

Hay 10 características predictoras (las 12 columnas menos cliente_id y compra_proximo_mes). X tiene forma 400, 10 e y tiene 400 etiquetas: 246 clientes con No (0) y 154 con Sí (1), es decir, un 61,5 % / 38,5 %, así que las clases están algo desbalanceadas.

### Ejercicio 6. División entrenamiento/prueba
Separa 80 % para entrenamiento y 20 % para prueba. Utiliza `random_state=42` y `stratify=y`. Comprueba tamaños.

In [6]:
# TODO: completar
X_train, X_test, y_train, y_test = train_test_split(
     X, y, test_size=0.2, random_state=42, stratify=y
)
# TODO: mostrar las dimensiones de los cuatro conjuntos

print("Entrenamiento :" ,X_train.shape, y_train.shape)
print("test  :" ,X_test.shape, y_test.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

Entrenamiento : (320, 10) (320,)
test  : (80, 10) (80,)
compra_proximo_mes
0    0.615625
1    0.384375
Name: proportion, dtype: float64
compra_proximo_mes
0    0.6125
1    0.3875
Name: proportion, dtype: float64


**Responde:** ¿Cuántas muestras hay en cada conjunto? ¿Para qué sirve `stratify=y`? ¿Por qué dividimos antes de ajustar transformaciones?

* **Número de muestras:** Hay 320 muestras en el conjunto de entrenamiento (80 %) y 80 muestras en el conjunto de test (20 %).
* **Uso de stratify=y:** Sirve para mantener en ambos subconjuntos exactamente la misma proporción de clases de la variable objetivo que en el dataset original, evitando que por azar un conjunto quede descompensado.
* **Dividir antes de transformar:** Se hace para evitar la fuga de datos. Si calculamos parámetros como medias, medianas o escalados usando todo el dataset, el modelo recibiría información estadística del conjunto de test durante el entrenamiento, lo que falsearía su capacidad real de generalización.

## Parte 3 - Imputación y encoding
### Ejercicio 7. Validación de edad
Sobre **copias** de `X_train` y `X_test`, sustituye edades superiores a 120 por `np.nan`. No imputes todavía: eso se hará en el pipeline. El umbral es una regla didáctica de validación del dominio.

In [7]:
X_train = X_train.copy()
X_test = X_test.copy()

# TODO: marcar edades > 120 como NaN en ambos conjuntos
print(X_train.loc[df["edad"]>120,["edad"]])
X_train.loc[df["edad"]>120,["edad"]] = np.nan
print(X_train["edad"].isna().sum())

print(X_test.loc[df["edad"]>120,["edad"]])
X_test.loc[df["edad"]>120,["edad"]]=np.nan
print(X_test["edad"].isna().sum())

# TODO: contar ausentes en edad en entrenamiento y prueba

     edad
137   195
1
Empty DataFrame
Columns: [edad]
Index: []
0


### Ejercicio 8. Estrategias de imputación
Utilizaremos mediana para numéricas y moda para categóricas. Crea ambos objetos `SimpleImputer` **sin ajustarlos todavía**.

In [8]:
# TODO: imputer_numerico = SimpleImputer(...)
imputer_numerico = SimpleImputer(strategy="median")

# TODO: imputer_categorico = SimpleImputer(...)
imputer_categorico = SimpleImputer(strategy="most_frequent")

| Variable | Estrategia | Justificación |
|---|---|---|
| edad | Mediana | Es una variable numérica y la mediana es robusta frente a valores atípicos (como el error detectado de 195 años). |
| antiguedad_meses | Mediana | Es una variable numérica y evita que posibles valores extremos distorsionen el valor imputado. |
| ingresos_anuales | Mediana | Es numérica y mantiene el valor central sin verse afectada por el tope mínimo de 9.000 € o valores muy altos. |
| gasto_mensual | Mediana | Es numérica y presenta asimetría a la derecha y outliers superiores (hasta 260 €), por lo que la mediana representa mejor el centro que la media. |
| dispositivo | Moda| Es una variable categórica nominal (texto), por lo que no se puede calcular media ni mediana y se rellena con la categoría más frecuente Móvil. |
| zona | Moda | Es una variable categórica nominal (texto) y la moda permite asignar la categoría más habitual sin alterar el tipo de dato. |

**¿Por qué no ajustar el imputador con entrenamiento y prueba a la vez?**

Respuesta: Porque se produciría fuga de datos (*data leakage*). Si ajustamos el imputador (`fit`) usando también el conjunto de prueba, el cálculo de la mediana o la moda incorporaría información estadística de los datos de test durante el entrenamiento, contaminando la evaluación y ofreciendo una estimación demasiado optimista e irreal del rendimiento del modelo ante datos nuevos.

### Ejercicio 9. One-Hot Encoding
Prueba `OneHotEncoder` con un DataFrame independiente. Configura `handle_unknown="ignore"` y `sparse_output=False`. Muestra matriz y nombres de columnas.

In [9]:
ejemplo = pd.DataFrame({
    "dispositivo": ["Móvil", "Ordenador", "Tablet", "Móvil"]
})

# TODO: encoder_nominal = OneHotEncoder(...)
encoder_nominal = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
# TODO: aplicar fit_transform a ejemplo
matriz_ejemplo = encoder_nominal.fit_transform(ejemplo)
# TODO: consultar get_feature_names_out()
print(encoder_nominal.get_feature_names_out())
pd.DataFrame(matriz_ejemplo, columns=encoder_nominal.get_feature_names_out())

['dispositivo_Móvil' 'dispositivo_Ordenador' 'dispositivo_Tablet']


,dispositivo_Móvil,dispositivo_Ordenador,dispositivo_Tablet
0,1.0,0.0,0.0
1,0.0,1.0,0.0
2,0.0,0.0,1.0
3,1.0,0.0,0.0


**Responde:** ¿Cuántas columnas se generan? ¿Qué representan? ¿Por qué no codificar Móvil=0, Ordenador=1 y Tablet=2 como categorías ordenadas?

Respuesta:
* **Columnas generadas:** Se generan 3 columnas, una por cada categoría única de la variable.
* **Qué representan:** Cada columna es una variable binaria (0.0 o 1.0) que indica presencia o ausencia: un 1.0 señala qué dispositivo utilizó ese registro concreto y 0.0 en las demás categorías.
* **Por qué no usar 0, 1 y 2:** Porque dispositivoes una variable categórica nominal sin orden ni jerarquía natural. Asignarle números consecutivos haría que los modelos matemáticos interpreten erróneamente que Tablet (2) > Ordenador (1) > Móvil (0) o que existe una distancia numérica entre ellos.

### Ejercicio 10. Codificación ordinal
Configura `OrdinalEncoder` para el orden Bronce, Plata, Oro, Platino. Comprueba su funcionamiento con un ejemplo independiente.

In [10]:
ejemplo_fidelidad = pd.DataFrame({
    "nivel_fidelidad": ["Oro", "Bronce", "Platino", "Plata"]
})

# TODO: encoder_ordinal = OrdinalEncoder(categories=[[...]])
encoder_ordinal = OrdinalEncoder(categories=[["Bronce", "Plata", "Oro", "Platino"]],
        handle_unknown="use_encoded_value",
        unknown_value=-1)
# TODO: transformar ejemplo_fidelidad y mostrar el resultado
resultado = encoder_ordinal.fit_transform(ejemplo_fidelidad)
print(resultado)
print(resultado.ravel())

[[2.]
 [0.]
 [3.]
 [1.]]
[2. 0. 3. 1.]


**Responde:** ¿Qué representan los códigos? ¿La distancia entre niveles consecutivos es necesariamente igual?

* **Qué representan los códigos:** Representan la posición jerárquica de cada nivel de fidelidad según el orden definido (Bronce = 0, Plata = 1, Oro = 2, Platino = 3). De este modo, el resultado [2., 0., 3., 1.] corresponde respectivamente a Oro, Bronce, Platino y Plata.
* **Distancia entre niveles:** No, en la realidad la distancia entre niveles consecutivos no es necesariamente igual (por ejemplo, el salto de requisitos o valor entre Oro y Platino puede ser mucho mayor que entre Bronce y Plata). Sin embargo, al codificarlos como 0, 1, 2, 3, los algoritmos matemáticos asumirán esa separación uniforme como una aproximación práctica para conservar el orden.

## Parte 4 - Escalado
### Ejercicio 11. Min-Max frente a Z-score
Aplica ambos escaladores al siguiente ejemplo. Redondea a tres decimales y completa la tabla.

In [11]:
datos = pd.DataFrame({"valor": [10, 20, 30, 40, 50]})

# TODO: crear MinMaxScaler y StandardScaler
minmax = MinMaxScaler()
zscore = StandardScaler()
# TODO: transformar datos con cada uno
comparacion = {
    "Original": datos["valor"],
    "Min-Max": minmax.fit_transform(datos).ravel(),
    "StandardScaler": zscore.fit_transform(datos).ravel()
}
# TODO: mostrar tabla comparativa
tabla = pd.DataFrame(comparacion).round(3)
tabla

,Original,Min-Max,StandardScaler
0,10,0.00,-1.414
1,20,0.25,-0.707
2,30,0.50,0.000
3,40,0.75,0.707
4,50,1.00,1.414


| Original | Min-Max | StandardScaler |
|---:|---:|---:|
| 10 | 0.00 | -1.414 |
| 20 | 0.25 | -0.707 |
| 30 | 0.50 | 0.000 |
| 40 | 0.75 | 0.707 |
| 50 | 1.00 | 1.414 |

**Responde:** ¿Qué rango da Min-Max en este ejemplo? ¿Qué valor toma la media tras estandarizar? ¿Pueden aparecer valores negativos? ¿Qué escalado elegirías para un algoritmo basado en distancias y por qué?

Respuesta:

* **Rango de Min-Max:** Devuelve valores comprendidos en el intervalo cerrado [0, 1] (de 0.00 a 1.00).
* **Media tras estandarizar:** Toma el valor 0 (0.000), ya que StandardScaler centra los datos restando la media original (30) y dividiendo entre la desviación estándar.
* **Valores negativos:** Sí, con StandardScaler aparecen valores negativos -1.414 y -0.707 para todos aquellos datos que estén por debajo de la media original.
* **Escalado para algoritmos basados en distancias (como KNN o SVM):** Elegiría StandardScaler porque centra todas las variables en media 0 y desviación estándar 1 sin comprimir la mayoría de los datos en un rango muy estrecho si existen valores atípicos (outliers), manteniendo proporciones equilibradas para el cálculo de distancias euclídeas.

### Ejercicio 12. Variables que escalaremos
Para esta práctica trataremos `satisfaccion` como numérica, aunque conceptualmente es una escala ordinal de 1 a 5. Es una simplificación que deberá revisarse en otros modelos.

In [12]:
numericas = [
    "edad", "antiguedad_meses", "ingresos_anuales",
    "gasto_mensual", "num_compras", "devoluciones", "satisfaccion"
]
nominal = ["dispositivo", "zona"]
ordinal = ["nivel_fidelidad"]

# Nos aseguramos de comprobar que existan si X_train está definido en el espacio de nombres
if 'X_train' in globals():
    columnas_definidas = numericas + nominal + ordinal
    columnas_en_train = list(X_train.columns)
    print("¿Todas las columnas definidas existen en X_train?:", set(columnas_definidas).issubset(set(columnas_en_train)))
    print("Columnas en X_train que no hemos clasificado:", set(columnas_en_train) - set(columnas_definidas))
else:
    print("Nota: X_train no está cargado en memoria todavía. Ejecuta las celdas anteriores primero.")

¿Todas las columnas definidas existen en X_train?: True
Columnas en X_train que no hemos clasificado: set()


**Responde:** ¿Qué variables tienen escalas más distintas? ¿Es obligatorio escalar para cualquier algoritmo? ¿Qué particularidad tiene `satisfaccion`?

**Respuesta:**
* **Escalas más distintas:** Las variables `ingresos_anuales` (que tiene valores de miles de euros) y `gasto_mensual` (valores de cientos) comparadas con `devoluciones` o `satisfaccion` (valores enteros pequeños de un solo dígito).
* **Obligatoriedad del escalado:** No es obligatorio para todos. Los algoritmos basados en árboles de decisión (como Decision Trees, Random Forests o Gradient Boosting) son invariantes a la escala. Sin embargo, para algoritmos basados en distancias (KNN, SVM) o en optimización por gradiente (Regresión Lineal/Logística, Redes Neuronales), el escalado es fundamental.
* **Particularidad de `satisfaccion`:** Es conceptualmente una variable cualitativa ordinal en escala Likert (1-5). Aunque no es continua, al tratarla como numérica en esta práctica, estamos asumiendo que la distancia entre cada nivel (ej. de 1 a 2, y de 4 a 5) es exactamente igual y tiene un sentido numérico directo.

## Parte 5 - Pipeline completo
### Ejercicio 13. Pipeline numérico
Construye un `Pipeline` que impute con mediana y después estandarice.

In [13]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Aseguramos la existencia de imputer_numerico por si no se ejecutó antes
if 'imputer_numerico' not in globals():
    imputer_numerico = SimpleImputer(strategy="median")

scaler_numerico = StandardScaler()

pipeline_numerico = Pipeline(steps=[
     ("imputer", imputer_numerico),
     ("scaler", scaler_numerico)
])

**¿Por qué imputamos antes de escalar en este pipeline?**

**Respuesta:** Porque el escalador (como `StandardScaler`) necesita calcular la media y la desviación estándar para transformar los datos. Si dejáramos valores ausentes (`NaN`), los cálculos matemáticos de estos parámetros fallarían o no se podrían aplicar correctamente sobre valores inexistentes. Además, muchos estimadores y transformadores de scikit-learn no aceptan valores vacíos como entrada.

### Ejercicio 14. Pipeline nominal
Imputa con moda y después aplica One-Hot Encoding (`handle_unknown="ignore"`, `sparse_output=False`).

In [14]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

# Aseguramos las dependencias locales
if 'imputer_categorico' not in globals():
    imputer_categorico = SimpleImputer(strategy="most_frequent")
if 'encoder_nominal' not in globals():
    encoder_nominal = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

pipeline_nominal = Pipeline(steps=[
    ("imputer", imputer_categorico),
    ("encoder", encoder_nominal)
])

### Ejercicio 15. Pipeline ordinal
Imputa con moda y codifica Bronce → Plata → Oro → Platino. Gestiona categorías desconocidas con `handle_unknown="use_encoded_value"` y `unknown_value=-1`.

In [15]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer

# Aseguramos el imputador
if 'imputer_categorico' not in globals():
    imputer_categorico = SimpleImputer(strategy="most_frequent")

pipeline_ordinal = Pipeline(steps=[
    ("imputer", imputer_categorico),
    ("encoder", OrdinalEncoder(
        categories=[["Bronce", "Plata", "Oro", "Platino"]],
        handle_unknown="use_encoded_value",
        unknown_value=-1
    ))
])

### Ejercicio 16. ColumnTransformer
Combina los tres pipelines para aplicar cada transformación únicamente a sus columnas.

In [16]:
from sklearn.compose import ColumnTransformer

# Definimos las variables por si acaso no se ejecutó la celda 12 en el kernel
numericas = ["edad", "antiguedad_meses", "ingresos_anuales", "gasto_mensual", "num_compras", "devoluciones", "satisfaccion"]
nominal = ["dispositivo", "zona"]
ordinal = ["nivel_fidelidad"]

preprocesador = ColumnTransformer(transformers=[
     ("num", pipeline_numerico, numericas),
     ("nom", pipeline_nominal, nominal),
     ("ord", pipeline_ordinal, ordinal)
])

### Ejercicio 17. Transformar entrenamiento y prueba
Ajusta el preprocesador **solo** con entrenamiento y aplícalo a prueba sin reajustarlo.

In [17]:
import numpy as np
import pandas as pd

# En caso de que no se hayan ejecutado las celdas de carga y split en tu sesión actual
if 'X_train' not in globals():
    df = pd.read_csv("clientes_ecommerce.csv")
    X = df.drop(columns=["cliente_id", "compra_proximo_mes"])
    y = df["compra_proximo_mes"].map({"No": 0, "Sí": 1})
    from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(
         X, y, test_size=0.2, random_state=42, stratify=y
    )
    # Aplicar la regla didáctica de validación del Ejercicio 7
    X_train = X_train.copy()
    X_test = X_test.copy()
    X_train.loc[X_train["edad"] > 120, "edad"] = np.nan
    X_test.loc[X_test["edad"] > 120, "edad"] = np.nan

X_train_preparado = preprocesador.fit_transform(X_train)
X_test_preparado = preprocesador.transform(X_test)

**Responde:** ¿Por qué no usar `fit_transform()` en prueba? ¿Qué aprende `fit()`? ¿Qué ocurre si aparece una categoría nominal nueva en prueba?

**Respuesta:**
* **No usar fit_transform en prueba:** Porque causaríamos fuga de datos (*data leakage*). Estaríamos calculando estadísticas (como medias, modas o categorías únicas) del conjunto de test, lo que falsearía la evaluación del modelo con datos que se supone que son nuevos y desconocidos.
* **Qué aprende fit():** El método `fit()` calcula e internaliza los parámetros de las transformaciones basándose exclusivamente en los datos de entrenamiento (por ejemplo: las medianas de las numéricas, las modas de las categóricas, y el mapeo de categorías únicas a columnas dummy).
* **Categoría nominal nueva en prueba:** Al configurar `OneHotEncoder(handle_unknown="ignore")`, si en test aparece una categoría nueva que no estaba en entrenamiento, se ignorará de manera segura codificándose con ceros en todas las columnas binarias de esa característica.

## Parte 6 - Validación e informe
### Ejercicio 18. Verificar matrices
Comprueba tamaños, ausentes, tipos numéricos, correspondencia con las etiquetas y nombres de características.

In [18]:
print("Dimensión de X_train_preparado:", X_train_preparado.shape)
print("Dimensión de X_test_preparado:", X_test_preparado.shape)

print("¿Quedan valores nulos en entrenamiento?:", np.isnan(X_train_preparado).any())
print("¿Quedan valores nulos en prueba?:", np.isnan(X_test_preparado).any())

print("Tipo de datos de la matriz preparada:", X_train_preparado.dtype)

# Verificar correspondencia con las etiquetas (longitud de filas)
print("¿Coinciden filas de entrenamiento con y_train?:", X_train_preparado.shape[0] == len(y_train))
print("¿Coinciden filas de prueba con y_test?:", X_test_preparado.shape[0] == len(y_test))

# Mostrar nombres de características
columnas_finales = preprocesador.get_feature_names_out()
print("Número de columnas resultantes:", len(columnas_finales))
print("Nombres de características resultantes:\n", columnas_finales)

Dimensión de X_train_preparado: (320, 16)
Dimensión de X_test_preparado: (80, 16)
¿Quedan valores nulos en entrenamiento?: False
¿Quedan valores nulos en prueba?: False
Tipo de datos de la matriz preparada: float64
¿Coinciden filas de entrenamiento con y_train?: True
¿Coinciden filas de prueba con y_test?: True
Número de columnas resultantes: 16
Nombres de características resultantes:
 ['num__edad' 'num__antiguedad_meses' 'num__ingresos_anuales'
 'num__gasto_mensual' 'num__num_compras' 'num__devoluciones'
 'num__satisfaccion' 'nom__dispositivo_Móvil' 'nom__dispositivo_Ordenador'
 'nom__dispositivo_Tablet' 'nom__zona_Centro' 'nom__zona_Este'
 'nom__zona_Norte' 'nom__zona_Oeste' 'nom__zona_Sur'
 'ord__nivel_fidelidad']


**Responde:** ¿Por qué aumentaron las columnas? ¿Coinciden las características en ambos conjuntos? ¿Quedan ausentes? ¿Están alineadas las etiquetas?

**Respuesta:**
* **Aumento de columnas:** Porque la codificación One-Hot aplicada a las variables nominales (`dispositivo` y `zona`) expande cada categoría única en su propia columna binaria.
* **Coincidencia de características:** Sí, puesto que el conjunto de test se transforma usando exactamente el preprocesador que fue ajustado con el conjunto de entrenamiento, asegurando que ambos tengan el mismo número y orden de columnas.
* **Ausentes:** No queda ningún valor nulo ya que imputamos previamente todos los ausentes con la mediana y la moda.
* **Alineación de etiquetas:** Sí, las filas resultantes de las matrices transformadas están perfectamente alineadas en orden e índice con las series de etiquetas `y_train` e `y_test`.

### Ejercicio 19. Reconstruir DataFrames
Convierte las matrices preparadas en `df_train_preparado` y `df_test_preparado`. Usa `preprocesador.get_feature_names_out()` y conserva los índices de X_train/X_test.

In [20]:
columnas_finales = preprocesador.get_feature_names_out()

df_train_preparado = pd.DataFrame(X_train_preparado, columns=columnas_finales, index=X_train.index)
df_test_preparado = pd.DataFrame(X_test_preparado, columns=columnas_finales, index=X_test.index)

print("Vista previa Entrenamiento Preparado")
display(df_train_preparado.head())
print("\n Vista previa Test Preparado")
display(df_test_preparado.head())

Vista previa Entrenamiento Preparado


,num__edad,num__antiguedad_meses,num__ingresos_anuales,num__gasto_mensual,num__num_compras,num__devoluciones,num__satisfaccion,nom__dispositivo_Móvil,nom__dispositivo_Ordenador,nom__dispositivo_Tablet,nom__zona_Centro,nom__zona_Este,nom__zona_Norte,nom__zona_Oeste,nom__zona_Sur,ord__nivel_fidelidad
360,-0.068723,-1.095565,0.651045,-1.086877,0.277945,-0.185205,-0.254293,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,3.0
356,-0.677904,1.129874,1.422884,-1.092939,0.277945,-1.069767,-0.254293,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
138,0.845048,-1.540653,0.907575,0.537957,-0.457116,-1.069767,1.495680,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
42,-1.408922,0.447406,0.476223,0.303390,1.380537,-1.069767,1.495680,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
216,-0.495150,1.456272,-0.493983,1.466735,0.645476,-1.069767,1.495680,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0



 Vista previa Test Preparado


,num__edad,num__antiguedad_meses,num__ingresos_anuales,num__gasto_mensual,num__num_compras,num__devoluciones,num__satisfaccion,nom__dispositivo_Móvil,nom__dispositivo_Ordenador,nom__dispositivo_Tablet,nom__zona_Centro,nom__zona_Este,nom__zona_Norte,nom__zona_Oeste,nom__zona_Sur,ord__nivel_fidelidad
388,0.053113,-1.540653,-0.994595,-0.887364,1.013006,1.583919,-0.254293,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
302,-0.495150,1.040856,-0.185324,-0.504414,-0.089586,-1.069767,-0.254293,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
21,1.697902,-1.036220,0.611217,-0.253507,-0.457116,1.583919,-0.254293,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,2.0
368,0.784130,-1.095565,0.373354,-0.176812,-1.559708,1.583919,0.620694,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,2.0
378,-0.312396,0.269371,-1.092628,1.067445,1.013006,-1.069767,1.495680,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
